# Mi loco estamos crazy
## real


In [13]:
import mne
import numpy as np
import os
import random
import matplotlib.pyplot as plt

In [14]:
def trim_leading_zeros(raw, threshold=1):
    """Trim leading samples where all EEG channels are zero or flat."""
    data, times = raw.get_data(picks='eeg', return_times=True)

    signal_magnitude = np.abs(data).sum(axis=0)
    first_valid = np.argmax(signal_magnitude > threshold)

    raw.crop(tmin=times[first_valid])
    return raw

def open_bdf(path):
    raw = mne.io.read_raw_bdf(
        path,
        preload=True,
        stim_channel="Status"
    )
    mapping = {
    "EEG 1": "Fz",
    "EEG 2": "C3",
    "EEG 3": "Cz",
    "EEG 4": "C4",
    "EEG 5": "Pz",
    "EEG 6": "PO7",
    "EEG 7": "Oz",
    "EEG 8": "PO8"
    }

    raw.rename_channels(mapping)
    raw.drop_channels(['ACC X', 'ACC Y', 'ACC Z', 'GYR X', 'GYR Y', 'GYR Z', 'BAT', 'CNT', 'VALID', 'DT'])

    raw = trim_leading_zeros(raw)
    
    return raw

def purity(seg):
    c = np.bincount(seg, minlength=3)
    return c.max()/c.sum()  #nos da el porentaje

In [15]:
#main

dir_name = 'datos/'
fnames = [f for f in os.listdir(dir_name) if f.endswith('.bdf')]
raw = mne.concatenate_raws([open_bdf(dir_name + f) for f in fnames])

print("Extrayendo eventos...")
events = mne.find_events(raw, stim_channel="Status")
print("IDs únicos:", np.unique(events[:, 2], return_counts=True))

sfreq = raw.info["sfreq"]
n_times = raw.n_times  #todas las muestras del registro

TASK_DUR = 1.5
task_samp = int(round(TASK_DUR * sfreq))

y_t = np.zeros(n_times, dtype=np.int16)  # 0 reposo lo establecemos como predeterminado
for samp, _, code in events:
    start = int(samp)
    end = min(start + task_samp, n_times)
    y_t[start:end] = int(code)  # 1 o 2

print("Estados (0/1/2) en todo el registro:", np.unique(y_t, return_counts=True))


# Ahora solo EEG (el print es para ver los canales que se recnocen como eeg, que son 18, al filtrar por nombre solo nos quedamos con los electrodos)

#for name, typ in zip(raw.ch_names, raw.get_channel_types()):
#    print(f"{name:>12}  {typ}")

#el filtro de los primeros 8

eeg_channels = ["Fz", "C3", "Cz", "C4", "Pz", "PO7", "Oz", "PO8"]
raw.pick_channels(eeg_channels)  # Ahora solo quedan estos 8 canales

#ventanas + etiquetas por ventana. Lo que hacee es dividir las ventanas en instantes (en 250Hz) 
raw_f = raw.copy().filter(8., 30., verbose=False)

sfreq = raw_f.info["sfreq"]
win_sec = 1.5    #tiempo de cada ventana
step_sec = 0.25  #timepo entre ventanas (se solapan, no se si dara problemas) 
win = int(round(win_sec * sfreq))  #instantes en una ventana
step = int(round(step_sec * sfreq))   #instantes entre ventanas

Xsig = raw_f.get_data()  # matriz (canales, muestras)
starts = np.arange(0, raw_f.n_times - win, step)  #donde n_times son el numero de muestras

# ventanas
windows = np.stack([Xsig[:, s:s+win] for s in starts], axis=0) 
#la ventana tiene la forma (num ventanas, canales, muestras por ventana)
# etiqueta por ventana, quien es el estado ganador. 
#La etiqueta es de cada ventana, no de cada muestra!!
#pero se etiquetan las muestras para dar un ganador de ventana

y_win = np.zeros(len(starts), dtype=np.int16) #esto es un vector vacío donde meter la etiqueta de cada ventana
for i, s in enumerate(starts):
    seg = y_t[s:s+win]   #aqui se elige desde el inicio 's' hasta el final de esa ventana 's+win' ya que win es la longitud de una ventana en instantes
    y_win[i] = np.bincount(seg, minlength=3).argmax()   #y aqui se cuentan cuantas hay de cada tipo y se elige el valor que más hay (0,1,2)

print("Estados por ventana:", np.unique(y_win, return_counts=True))   #print de cuantas ventanas hay de cada clase


#aqui estamos solo quedandonos con las ventanas en las que el 90% o más es la misma respuesta, las demás las descartamos...
mask = np.array([purity(y_t[s:s+win]) >= 0.7 for s in starts])
x = windows[mask]
y = y_win[mask]
print("Train:", x.shape, np.unique(y, return_counts=True))

np.savez("datos_procesados.npz", x=x, y=y)

Extracting BDF parameters from datos/Lucas_FlexoExtension_Motor_Der10.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 30550  =      0.000 ...   122.200 secs...
Extracting BDF parameters from datos/Lucas_FlexoExtension_Motor_Der11.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 27761  =      0.000 ...   111.044 secs...
Extracting BDF parameters from datos/Lucas_FlexoExtension_Motor_Der12.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 27414  =      0.000 ...   109.656 secs...
Extracting BDF parameters from datos/Lucas_FlexoExtension_Motor_Der13.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 27597  =      0.000 ...   110.388 secs...
Extracting BDF parameters from datos/Lucas_FlexoExtension_Motor_Der14.bdf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 27872  =      0.000 ...   111.488 secs...
Extracting